# Time Channel Data Smoothing

- Load and normalize the NE ATEM data.
- Bin each flight line at 50 m spacing.
- Estimate channel reliability from binned relative error.
- Smooth each time channel with KDTree neighbors, IDW distance weights, and reliability weights.
- Keep the original value where the effective neighbor count is too small.
- Compare original and smoothed data on maps and line profiles.
- Aggregate residual RMS by flight line and compare high-score lines with inversion depth slices.


In [ ]:
import json
import dill

import numpy as np
import pandas as pd
from rich import print
from matplotlib import pyplot as plt
from matplotlib import rcParams as rc
from matplotlib.colors import LogNorm
from ipywidgets import widgets
from ipywidgets import VBox, interactive_output
from scipy.spatial import cKDTree as KDTree
from simpeg.electromagnetics.utils.em1d_utils import get_vertical_discretization

rc["font.family"] = "Times New Roman"
rc["font.size"] = 14
rc["figure.figsize"] = (6, 4)
rc["axes.grid"] = True


## Load Configuration

Select the time channels used in the previous inversion workflow.


In [ ]:
istart_channel: int = 3
iend_channel: int = 25

with open("../../atem/data/atem.json") as f:
    conf = json.load(f)
times = np.asarray(conf["channels"])[istart_channel:iend_channel] * 1e-6
n_turns = conf["n_turns"]

dheader = [f"zoff30[{i}]" for i in range(istart_channel, iend_channel)]
time_labels = [f"{name} / {time:.2e} s" for name, time in zip(dheader, times)]


## Load And Normalize Data

Read the NE survey data and apply the same normalization used in notebook 06.


In [ ]:
area: str = "NE"
path: str = f"../../atem/data/11-024_Alberta_{area}.csv"
picker = [
    "Line",
    "bheight",
    "TranPeak",
    "x_wgs84",
    "y_wgs84",
    "flight",
    "pwrline",
    "dtm",
] + dheader

raws = pd.read_csv(path)[picker]
xy_raw = raws[["x_wgs84", "y_wgs84"]].to_numpy()
normalizer = (-1e-9) / (raws["TranPeak"].values * n_turns).reshape(-1, 1)
raws[dheader] = raws[dheader] * normalizer
line_no = list(raws["Line"].unique())


## Check Missing Coordinates

Identify flight lines with missing y coordinates before binning.


In [ ]:
nan_list = raws[raws["y_wgs84"].isna()]["Line"].unique()
print(nan_list)

counts = []
for line_n in nan_list:
    count = raws[raws["Line"] == line_n]["y_wgs84"].isna().sum()
    counts.append(count)
print(counts)

plt.figure(figsize=(6, 6))
plt.scatter(xy_raw[:, 0], xy_raw[:, 1], s=1, c="lightgray", label="All data")
for i, line_n in enumerate(nan_list):
    test_x = raws[raws["Line"] == line_n]["x_wgs84"]
    test_y = raws[raws["Line"] == line_n]["y_wgs84"]
    plt.scatter(test_x, test_y, s=1, c=f"C{i}", label=line_n)
plt.xlabel("x_wgs84")
plt.ylabel("y_wgs84")
plt.title("Lines having NaN values in y_wgs84")
plt.legend(markerscale=4, loc="upper right")
plt.tight_layout()


## Clean And Bin Data

Remove missing y coordinates and bin each line at 50 m spacing.


In [ ]:
raws.dropna(subset=["y_wgs84"], inplace=True)
raws.fillna(1e-20, inplace=True)

dx = 50.0
values = []
values_std = []
soundings = []
istart = 0
iend = len(line_no)

for i_line, line in enumerate(line_no[istart:iend]):
    df_line = raws[raws["Line"] == line].copy()

    xy_line = df_line[["x_wgs84", "y_wgs84"]].to_numpy()
    distance = np.sqrt(((xy_line - xy_line[0, :]) ** 2).sum(axis=1))
    max_distance = distance.max()

    if max_distance % dx == 0:
        n_sounding = int(max_distance / dx)
    else:
        n_sounding = int(np.round(max_distance / dx) + 1)

    bins = np.arange(n_sounding) * dx
    df_line.insert(0, "distance", distance)
    df_line["bin"] = pd.cut(df_line["distance"], bins=bins)

    binned = (
        df_line.groupby("bin", observed=False)
        [["distance"] + picker[1:]]
        .mean()
    )
    binned.insert(0, "Line", line)

    binned_std = (
        df_line.groupby("bin", observed=False)
        [["bheight"] + dheader]
        .std()
    )

    values.append(binned.values)
    values_std.append(binned_std.values)
    soundings.append(n_sounding)

df_data_binned = pd.DataFrame(
    data=np.vstack(values),
    columns=["Line", "distance"] + picker[1:],
)
df_data_std_binned = pd.DataFrame(
    data=np.vstack(values_std),
    columns=["bheight"] + dheader,
)

valid_xy = np.isfinite(df_data_binned[["x_wgs84", "y_wgs84"]].values.astype(float)).all(axis=1)
if not valid_xy.all():
    print(f"Dropping {(~valid_xy).sum():,} empty binned rows before mapping.")
    df_data_binned = df_data_binned.loc[valid_xy].reset_index(drop=True)
    df_data_std_binned = df_data_std_binned.loc[valid_xy].reset_index(drop=True)

xy_binned = df_data_binned[["x_wgs84", "y_wgs84"]].values.astype(float)
data_binned = df_data_binned[dheader].values.astype(float)
data_to_plot = -data_binned
plot_skip = 5
plot_indices = np.arange(0, xy_binned.shape[0], plot_skip)

print(f"Binned soundings: {len(df_data_binned):,}")
print(f"Time channels: {data_binned.shape[1]}")
print(f"Plotting every {plot_skip} sounding(s): {len(plot_indices):,} points")


## Data Reliability Weights

Estimate channel-wise uncertainty from bin statistics and use reliable data more strongly during smoothing.


In [ ]:
relative_error_cutoff = 0.03
minimum_relative_uncertainty = 0.05
floor_fraction = 0.05
noise_floor = 5e-9 / (df_data_binned["TranPeak"].values.astype(float) * n_turns) * floor_fraction

data_std_binned = df_data_std_binned[dheader].values.astype(float)
data_rerr = np.divide(
    data_std_binned,
    np.abs(data_binned),
    out=np.full_like(data_std_binned, np.inf, dtype=float),
    where=np.abs(data_binned) > 0,
)
bad_data_mask = data_rerr > relative_error_cutoff

base_uncertainty = np.maximum(data_std_binned, np.abs(data_binned) * minimum_relative_uncertainty)
data_uncertainty = base_uncertainty + noise_floor[:, None]
data_uncertainty[bad_data_mask] = np.inf

data_reliability_weight = np.divide(
    1.0,
    data_uncertainty**2,
    out=np.zeros_like(data_uncertainty, dtype=float),
    where=np.isfinite(data_uncertainty) & (data_uncertainty > 0),
)

active_data_fraction = np.isfinite(data_uncertainty).mean()
print(f"Active data fraction: {active_data_fraction * 100:.1f}%")
print(f"Bad data fraction: {bad_data_mask.mean() * 100:.1f}%")

pd.DataFrame(
    {
        "channel": dheader,
        "time_s": times,
        "active_fraction": np.isfinite(data_uncertainty).mean(axis=0),
        "median_relative_error": np.nanmedian(data_rerr, axis=0),
    }
)


## Original Time-Channel Map

Display the binned data in x-y space for a selected time channel.


In [ ]:
def get_channel_limits(values, i_time, lower=2, upper=98):
    channel_values = values[:, i_time]
    finite = np.isfinite(channel_values)
    if not finite.any():
        return None, None
    return np.nanpercentile(channel_values[finite], [lower, upper])


map_line_options = [str(line) for line in pd.unique(df_data_binned["Line"])]


def plot_original_time_channel(i_channel, line_name):
    i_time = i_channel - istart_channel
    vmin, vmax = get_channel_limits(data_to_plot, i_time)
    line_mask = df_data_binned["Line"].astype(str).values == str(line_name)
    background_indices = plot_indices[~line_mask[plot_indices]]

    fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    out = ax.scatter(
        xy_binned[background_indices, 0],
        xy_binned[background_indices, 1],
        c=data_to_plot[background_indices, i_time],
        s=1,
        cmap="turbo",
        vmin=vmin,
        vmax=vmax,
        edgecolors="none",
    )
    ax.scatter(
        xy_binned[line_mask, 0],
        xy_binned[line_mask, 1],
        c=data_to_plot[line_mask, i_time],
        s=12,
        cmap="turbo",
        vmin=vmin,
        vmax=vmax,
        linewidths=0.15,
    )
    colorbar_mappable = plt.cm.ScalarMappable(norm=plt.Normalize(vmin=vmin, vmax=vmax), cmap="turbo")
    colorbar_mappable.set_array([])
    cb = plt.colorbar(colorbar_mappable, ax=ax, orientation="horizontal", fraction=0.05, pad=0.08)
    cb.set_label("-dB/dt")
    ax.set_xlabel("x_wgs84")
    ax.set_ylabel("y_wgs84")
    ax.set_aspect(1)
    ax.set_title(f"Original data: {time_labels[i_time]}, Line {line_name}")
    plt.tight_layout()
    plt.show()

original_time_slider = widgets.IntSlider(
    min=istart_channel,
    max=iend_channel - 1,
    step=1,
    value=istart_channel,
    description="zoff30:",
    continuous_update=False,
)
original_line_slider = widgets.SelectionSlider(
    options=map_line_options,
    value=map_line_options[0],
    description="Line:",
    continuous_update=False,
)
original_output = interactive_output(
    plot_original_time_channel,
    {"i_channel": original_time_slider, "line_name": original_line_slider},
)

VBox([original_time_slider, original_line_slider, original_output])


## KDTree Smoothing

Smooth each sounding with nearby soundings using distance weights and data reliability weights.

- `idw_power` controls the spatial weighting: $1 / (distance + epsilon)^{power}$.
- Early channels use slightly lower power, so they are smoothed more strongly.
- If the effective neighbor count is too small, the original value is retained.


In [ ]:
# Smaller k and larger IDW power make the smoothing weaker.
k_nearest_points = 50
idw_epsilon = 1
idw_power_early = 1.5
idw_power_late = 2.0
idw_power_by_time = np.linspace(idw_power_early, idw_power_late, data_binned.shape[1])

n_neighbors = min(k_nearest_points, xy_binned.shape[0])
smoothing_chunk_size = 50_000
min_effective_neighbors = 5.0
tree = KDTree(xy_binned)
smoothed_data = np.empty_like(data_binned)

for start in range(0, xy_binned.shape[0], smoothing_chunk_size):
    end = min(start + smoothing_chunk_size, xy_binned.shape[0])
    data_chunk = data_binned[start:end]
    distances, neighbor_indices = tree.query(xy_binned[start:end], k=n_neighbors)

    if n_neighbors == 1:
        distances = distances[:, None]
        neighbor_indices = neighbor_indices[:, None]

    neighbor_data = data_binned[neighbor_indices]
    spatial_weights_by_time = 1.0 / ((distances[:, :, None] + idw_epsilon) ** idw_power_by_time[None, None, :])
    total_weights = spatial_weights_by_time * data_reliability_weight[neighbor_indices]
    weight_sum = np.sum(total_weights, axis=1)
    weighted_sum = np.sum(total_weights * neighbor_data, axis=1)
    smoothed_chunk = np.divide(
        weighted_sum,
        weight_sum,
        out=np.full_like(weighted_sum, np.nan, dtype=float),
        where=weight_sum > 0,
    )

    weight_square_sum = np.sum(total_weights**2, axis=1)
    effective_neighbor_count = np.divide(
        weight_sum**2,
        weight_square_sum,
        out=np.zeros_like(weight_sum, dtype=float),
        where=weight_square_sum > 0,
    )
    insufficient_neighbors = effective_neighbor_count < min_effective_neighbors
    smoothed_chunk[insufficient_neighbors] = data_chunk[insufficient_neighbors]
    smoothed_data[start:end] = smoothed_chunk

smoothed_data_to_plot = -smoothed_data

print(f"KDTree neighbors: {n_neighbors}")
print(f"IDW epsilon: {idw_epsilon}")
print(f"IDW power range: {idw_power_by_time[0]:.2f} to {idw_power_by_time[-1]:.2f}")
print("Smoothing weights: spatial IDW x inverse-variance data reliability")
print(f"Smoothing chunk size: {smoothing_chunk_size:,}")
print(f"Minimum effective neighbors: {min_effective_neighbors}")

pd.DataFrame(
    {
        "channel": dheader,
        "time_s": times,
        "idw_power": idw_power_by_time,
    }
)


## Original And Smoothed Slices

Compare original and smoothed maps by time channel.


In [ ]:
def plot_time_channel_comparison(i_channel, line_name):
    i_time = i_channel - istart_channel
    values_pair = np.r_[data_to_plot[:, i_time], smoothed_data_to_plot[:, i_time]]
    finite = np.isfinite(values_pair)
    vmin, vmax = np.nanpercentile(values_pair[finite], [2, 98])
    line_mask = df_data_binned["Line"].astype(str).values == str(line_name)
    background_indices = plot_indices[~line_mask[plot_indices]]

    fig, axs = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)
    titles = ["Original", "KDTree smoothed"]
    arrays = [data_to_plot[:, i_time], smoothed_data_to_plot[:, i_time]]

    for ax, title, values_i in zip(axs, titles, arrays):
        ax.scatter(
            xy_binned[background_indices, 0],
            xy_binned[background_indices, 1],
            c=values_i[background_indices],
            s=1,
            cmap="turbo",
            vmin=vmin,
            vmax=vmax,
            edgecolors="none",
        )
        ax.scatter(
            xy_binned[line_mask, 0],
            xy_binned[line_mask, 1],
            c=values_i[line_mask],
            s=12,
            cmap="turbo",
            vmin=vmin,
            vmax=vmax,
            linewidths=0.15,
        )
        ax.scatter(
            xy_binned[line_mask, 0][0],
            xy_binned[line_mask, 1][0],
            c='k',
            s=50,
            marker="*",
            linewidths=0.15,
        )
        ax.set_xlabel("x_wgs84")
        ax.set_ylabel("y_wgs84")
        ax.set_aspect(1)
        ax.set_title(f"{title}: {time_labels[i_time]}, Line {line_name}")

    colorbar_mappable = plt.cm.ScalarMappable(norm=plt.Normalize(vmin=vmin, vmax=vmax), cmap="turbo")
    colorbar_mappable.set_array([])
    cb = fig.colorbar(colorbar_mappable, ax=axs, orientation="horizontal", fraction=0.05, pad=0.08)
    cb.set_label("dB/dt")
    plt.show()

comparison_time_slider = widgets.IntSlider(
    min=istart_channel,
    max=iend_channel - 1,
    step=1,
    value=istart_channel,
    description="zoff30:",
    continuous_update=False,
)
comparison_line_slider = widgets.SelectionSlider(
    options=map_line_options,
    value=map_line_options[0],
    description="Line:",
    continuous_update=False,
)
comparison_output = interactive_output(
    plot_time_channel_comparison,
    {"i_channel": comparison_time_slider, "line_name": comparison_line_slider},
)

VBox([comparison_time_slider, comparison_line_slider, comparison_output])


## Line Comparison

Compare original and smoothed data along a selected flight line.


In [ ]:
line_options = [str(line) for line in pd.unique(df_data_binned["Line"])]


def normalize_line_monitor(values):
    values = np.asarray(values, dtype=float)
    normalized = np.full_like(values, np.nan, dtype=float)
    finite = np.isfinite(values)
    if not finite.any():
        return normalized

    vmin = np.nanmin(values[finite])
    vmax = np.nanmax(values[finite])
    if np.isclose(vmax, vmin):
        normalized[finite] = 0.5
    else:
        normalized[finite] = (values[finite] - vmin) / (vmax - vmin)
    return normalized


def plot_line_original_smoothed(line_name, i_channel, show_all_channels):
    line_mask = df_data_binned["Line"].astype(str).values == str(line_name)
    if not line_mask.any():
        raise ValueError(f"No data found for line {line_name}")

    distances = df_data_binned.loc[line_mask, "distance"].values.astype(float)
    sort_idx = np.argsort(distances)
    distances = distances[sort_idx]

    original_line = -data_binned[line_mask, :][sort_idx, :]
    smoothed_line = -smoothed_data[line_mask, :][sort_idx, :]
    difference_line = original_line - smoothed_line
    bheight_line = df_data_binned.loc[line_mask, "bheight"].values.astype(float)[sort_idx]
    pwrline_line = df_data_binned.loc[line_mask, "pwrline"].values.astype(float)[sort_idx]
    bheight_norm = normalize_line_monitor(bheight_line)
    pwrline_norm = normalize_line_monitor(pwrline_line)

    fig, ax = plt.subplots(1, 1, figsize=(14, 5))

    if show_all_channels:
        for i_time in range(data_binned.shape[1]):
            original_label = "Original" if i_time == 0 else None
            smoothed_label = "Smoothed" if i_time == 0 else None
            ax.semilogy(
                distances,
                np.abs(original_line[:, i_time]),
                "k-",
                linewidth=0.7,
                alpha=0.35,
                label=original_label,
            )
            ax.semilogy(
                distances,
                np.abs(smoothed_line[:, i_time]),
                "C0--",
                linewidth=0.7,
                alpha=0.55,
                label=smoothed_label,
            )
        ax.set_ylabel("|-dB/dt|")
        ax.set_title(f"Original vs smoothed data: Line {line_name}")
    else:
        i_time = i_channel - istart_channel
        ax.plot(distances, original_line[:, i_time], "k-", linewidth=1.2, label="Original")
        ax.plot(distances, smoothed_line[:, i_time], "C0--", linewidth=1.2, label="Smoothed")
        ax.plot(distances, difference_line[:, i_time], "C3-", linewidth=0.8, alpha=0.8, label="Original - smoothed")
        ax.axhline(0, color="gray", linewidth=0.8)
        ax.set_ylabel("-dB/dt")
        ax.set_title(f"{time_labels[i_time]}: Line {line_name}")

    ax_monitor = ax.twinx()
    ax_monitor.plot(distances, bheight_norm, color="C2", linewidth=1.1, alpha=0.9, label="bheight (norm.)")
    ax_monitor.plot(distances, pwrline_norm, color="C4", linewidth=1.1, alpha=0.9, label="pwrline (norm.)")
    ax_monitor.set_ylabel("Normalized monitor value")
    ax_monitor.set_ylim(-0.05, 1.05)
    ax_monitor.grid(False)

    handles, labels = ax.get_legend_handles_labels()
    monitor_handles, monitor_labels = ax_monitor.get_legend_handles_labels()
    ax.legend(handles + monitor_handles, labels + monitor_labels, loc="upper right")

    ax.set_xlabel("Distance along line (m)")
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

line_comparison_slider = widgets.SelectionSlider(
    options=line_options,
    value=line_options[0],
    description="Line:",
)
line_channel_slider = widgets.IntSlider(
    min=istart_channel,
    max=iend_channel - 1,
    step=1,
    value=istart_channel,
    description="zoff30:",
    continuous_update=False,
)
line_all_channels_checkbox = widgets.Checkbox(
    value=True,
    description="All channels",
)
line_comparison_output = interactive_output(
    plot_line_original_smoothed,
    {
        "line_name": line_comparison_slider,
        "i_channel": line_channel_slider,
        "show_all_channels": line_all_channels_checkbox,
    },
)

VBox([line_comparison_slider, line_channel_slider, line_all_channels_checkbox, line_comparison_output])


## Line-Level RMS Scores

Compute relative residual RMS values and aggregate them by flight line.


In [ ]:
valid_residual_channels = np.isfinite(data_binned) & np.isfinite(smoothed_data)
relative_denominator_floor = np.nanpercentile(np.abs(smoothed_data), 5, axis=0)
relative_denominator = np.abs(smoothed_data) + relative_denominator_floor[None, :]
relative_residual = np.divide(
    data_binned - smoothed_data,
    relative_denominator,
    out=np.full_like(data_binned, np.nan, dtype=float),
    where=valid_residual_channels & (relative_denominator > 0),
)

valid_channel_count = np.isfinite(relative_residual).sum(axis=1)
sounding_residual_rms = np.sqrt(np.nanmean(relative_residual**2, axis=1))
sounding_residual_rms[valid_channel_count == 0] = np.nan

line_residual_threshold = 0.2
line_score_df = df_data_binned[["Line", "distance", "x_wgs84", "y_wgs84"]].copy()
line_score_df["sounding_residual_rms"] = sounding_residual_rms
line_score_df["valid_channel_count"] = valid_channel_count


def rms(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    return np.sqrt(np.mean(values**2))

line_l2_summary = (
    line_score_df.groupby("Line")
    .agg(
        n_sounding=("sounding_residual_rms", "size"),
        n_valid=("sounding_residual_rms", lambda x: np.isfinite(x).sum()),
        line_l2_rms=("sounding_residual_rms", rms),
        line_l2_median=("sounding_residual_rms", "median"),
        line_l2_p90=("sounding_residual_rms", lambda x: np.nanpercentile(x, 90)),
        line_l2_p95=("sounding_residual_rms", lambda x: np.nanpercentile(x, 95)),
        line_l2_max=("sounding_residual_rms", "max"),
        fraction_above_line_residual_threshold=("sounding_residual_rms", lambda x: np.nanmean(x > line_residual_threshold)),
        median_valid_channel_count=("valid_channel_count", "median"),
        x_center=("x_wgs84", "mean"),
        y_center=("y_wgs84", "mean"),
        distance_min=("distance", "min"),
        distance_max=("distance", "max"),
    )
    .reset_index()
)
line_l2_summary["valid_fraction"] = line_l2_summary["n_valid"] / line_l2_summary["n_sounding"]

line_l2_summary = line_l2_summary.sort_values("line_l2_rms", ascending=False).reset_index(drop=True)
print("Line score definition: RMS of (D - S) / (abs(S) + channel_floor), aggregated by line")
line_l2_summary.head(15)


## Inversion Depth Slice Setup

Prepare gridded depth slices from the saved inversion model for comparison with line-level RMS maps.


In [ ]:
inv_results_path = "../data/inv_results_atem_full.pik"
with open(inv_results_path, "rb") as f:
    outDict = dill.load(f)
iterations = len(outDict.keys())

topography_inv = df_data_binned[["x_wgs84", "y_wgs84", "dtm"]].values.astype(float)
thickness = get_vertical_discretization(21, 2, 1.17)
hz = np.r_[thickness, thickness[-1]]
depth_edges = np.r_[0.0, np.cumsum(hz)]
depth_centers = 0.5 * (depth_edges[:-1] + depth_edges[1:])
n_layer = len(hz)
n_sounding_inv = topography_inv.shape[0]

expected_model_size = n_sounding_inv * n_layer
rho_layers_by_iteration = {}
for i_iteration, result in outDict.items():
    model = np.asarray(result["m"])
    if model.size != expected_model_size:
        raise ValueError(
            f"Iteration {i_iteration} model size {model.size} does not match "
            f"n_sounding_inv x n_layer = {expected_model_size}."
        )
    rho_layers_by_iteration[i_iteration] = (1.0 / np.exp(model)).reshape((n_sounding_inv, n_layer))

# Grid and interpolation settings for inversion depth slices.
inv_map_dx = 100.0
inv_map_dy = 100.0
inv_x_pad = 1000.0
inv_y_pad = 1000.0
inv_k_nearest_points = 100
inv_max_distance = 500.0
inv_idw_power = 0.0

xy_inv = topography_inv[:, :2]
inv_xmin, inv_xmax = xy_inv[:, 0].min() - inv_x_pad, xy_inv[:, 0].max() + inv_x_pad
inv_ymin, inv_ymax = xy_inv[:, 1].min() - inv_y_pad, xy_inv[:, 1].max() + inv_y_pad

inv_nx = int((inv_xmax - inv_xmin) / inv_map_dx)
inv_ny = int((inv_ymax - inv_ymin) / inv_map_dy)
inv_x_map = np.arange(inv_nx) * inv_map_dx + inv_xmin
inv_y_map = np.arange(inv_ny) * inv_map_dy + inv_ymin
INV_X_map, INV_Y_map = np.meshgrid(inv_x_map, inv_y_map)

inv_map_tree = KDTree(xy_inv)
inv_query_points = np.c_[INV_X_map.ravel(), INV_Y_map.ravel()]
inv_distances_idw, inv_indices_idw = inv_map_tree.query(inv_query_points, k=int(inv_k_nearest_points))

inv_idw_epsilon = min(inv_map_dx, inv_map_dy)
inv_depth_slice_idw_weights = 1.0 / ((inv_distances_idw + inv_idw_epsilon) ** inv_idw_power)
inv_nearest_distance, _ = inv_map_tree.query(inv_query_points, k=1)
inv_mask_outside_data = inv_nearest_distance > inv_max_distance

print(f"Loaded inversion iterations: {iterations}")
print(f"Depth layers: {n_layer}")


## Observed Time-Channel Map And Inversion Depth Slice

Compare high-RMS flight lines on the observed time-domain data and the inversion depth slice.


In [ ]:
def get_resistivity_depth_grid(i_iteration, i_depth):
    values = rho_layers_by_iteration[i_iteration][:, i_depth]
    values_grid = (
        np.sum(inv_depth_slice_idw_weights * values[inv_indices_idw], axis=1)
        / np.sum(inv_depth_slice_idw_weights, axis=1)
    )
    values_grid[inv_mask_outside_data] = np.nan
    return values_grid.reshape(INV_X_map.shape)


def plot_observed_channel_and_depth_slice(i_iteration, i_depth, i_channel, top_n_lines):
    top_lines = line_l2_summary.head(top_n_lines)["Line"].astype(str).tolist()
    top_line_mask = df_data_binned["Line"].astype(str).isin(top_lines).values
    observed_values = data_to_plot[:, i_channel]
    observed_vmin, observed_vmax = get_channel_limits(data_to_plot, i_channel)
    rho_grid = get_resistivity_depth_grid(i_iteration, i_depth)
    rho_point_values = rho_layers_by_iteration[i_iteration][:, i_depth]

    fig, axs = plt.subplots(1, 2, figsize=(15, 7), constrained_layout=True)
    ax_data, ax_rho = axs

    out_data = ax_data.scatter(
        xy_binned[plot_indices, 0],
        xy_binned[plot_indices, 1],
        c=observed_values[plot_indices],
        s=2,
        cmap="turbo",
        vmin=observed_vmin,
        vmax=observed_vmax,
        edgecolors="none",
        alpha=0.5,
    )
    ax_data.scatter(
        xy_binned[top_line_mask, 0],
        xy_binned[top_line_mask, 1],
        c=observed_values[top_line_mask],
        s=12,
        cmap="turbo",
        vmin=observed_vmin,
        vmax=observed_vmax,
        linewidths=0.08,
        alpha=1.0,
        label=f"Top {top_n_lines} high-RMS lines",
    )
    cb_data = plt.colorbar(out_data, ax=ax_data, orientation="horizontal", fraction=0.05, pad=0.08)
    cb_data.set_label(f"Observed {dheader[i_channel]}")
    ax_data.set_xlabel("x_wgs84")
    ax_data.set_ylabel("y_wgs84")
    ax_data.set_aspect(1)
    ax_data.set_title(f"Observed data: {time_labels[i_channel]}")
    # ax_data.legend(markerscale=2, loc="upper right")

    out_rho_grid = ax_rho.pcolormesh(
        inv_x_map,
        inv_y_map,
        rho_grid,
        cmap="turbo",
        norm=LogNorm(vmin=8, vmax=60),
        shading="auto",
        alpha=0.5,
    )
    ax_rho.plot(xy_inv[:, 0], xy_inv[:, 1], "k,", alpha=0.12)
    ax_rho.scatter(
        xy_binned[top_line_mask, 0],
        xy_binned[top_line_mask, 1],
        c=rho_point_values[top_line_mask],
        s=12,
        cmap="turbo",
        norm=LogNorm(vmin=8, vmax=60),
        linewidths=0.05,
        label=f"Top {top_n_lines} high-RMS lines",
    )
    cb_rho = plt.colorbar(out_rho_grid, ax=ax_rho, orientation="horizontal", fraction=0.05, pad=0.08)
    cb_rho.set_label("Resistivity (ohm-m)")
    ax_rho.set_xlabel("x_wgs84")
    ax_rho.set_ylabel("y_wgs84")
    ax_rho.set_aspect(1)
    ax_rho.set_title(f"Inversion depth slice: iteration {i_iteration}, depth {depth_centers[i_depth]:.1f} m")
    ax_rho.legend(markerscale=2, loc="upper right")

    plt.show()
    display(line_l2_summary.head(top_n_lines)[["Line"]])


line_rms_iteration_slider = widgets.IntSlider(
    min=1,
    max=iterations,
    step=1,
    value=iterations,
    description="Iteration:",
    continuous_update=False,
)
line_rms_depth_slider = widgets.SelectionSlider(
    options=[(f"{i}: {depth_centers[i]:.1f} m", i) for i in range(n_layer)],
    value=0,
    description="Depth:",
    continuous_update=False,
)
line_rms_channel_slider = widgets.SelectionSlider(
    options=[(label, i) for i, label in enumerate(time_labels)],
    value=0,
    description="Channel:",
    continuous_update=False,
)
line_rms_top_n_slider = widgets.IntSlider(
    min=1,
    max=50,
    step=1,
    value=50,
    description="Top N:",
    continuous_update=False,
)
line_rms_depth_output = interactive_output(
    plot_observed_channel_and_depth_slice,
    {
        "i_iteration": line_rms_iteration_slider,
        "i_depth": line_rms_depth_slider,
        "i_channel": line_rms_channel_slider,
        "top_n_lines": line_rms_top_n_slider,
    },
)

VBox([
    line_rms_iteration_slider,
    line_rms_depth_slider,
    line_rms_channel_slider,
    line_rms_top_n_slider,
    line_rms_depth_output,
])
